# Step 5: Embed the item pool

This notebook embeds every image in the Kaggle "Fashion Product Images (Small)" dataset (~44k items) using the same FashionCLIP image encoder from notebook 02, and caches the resulting vectors to `data/embeddings/`. This only needs to run once -- every later step (recommendations, the usage classifier) will just load the saved result.

## Load the item metadata

Same as notebook 01: load `styles.csv`, skipping the handful of malformed rows. We point directly at the cached dataset folder rather than calling `kagglehub.dataset_download()` again, since that function checks the network for the latest version even when a copy is already cached locally -- and we don't need that check here.

In [5]:
from pathlib import Path
import pandas as pd

dataset_root = Path.home() / ".cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1"
images_dir = dataset_root / "images"
styles_csv = dataset_root / "styles.csv"

df = pd.read_csv(styles_csv, on_bad_lines="skip")
print("Metadata rows:", df.shape[0])

Metadata rows: 44424


## Match metadata rows to actual image files

Each image is named `<id>.jpg`, where `id` matches the `id` column in `styles.csv`. Real-world datasets are rarely perfectly clean -- some metadata rows may be missing their image file, or vice versa -- so we filter down to the intersection: only rows that have a matching file on disk. This keeps everything aligned for later steps, where we'll need to map an embedding back to its item's `usage` label, category, etc.

In [2]:
existing_image_ids = {int(p.stem) for p in images_dir.glob("*.jpg")}

df = df[df["id"].isin(existing_image_ids)].reset_index(drop=True)
print("Rows with a matching image file:", df.shape[0])

item_ids = df["id"].tolist()
item_paths = [images_dir / f"{item_id}.jpg" for item_id in item_ids]

Rows with a matching image file: 44419


## Load FashionCLIP

Same model as notebook 02 -- this notebook is self-contained, so it loads independently. If you already ran notebook 02 in this session, the weights are cached locally and this will be fast.

In [3]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "patrickjohncyh/fashion-clip"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("Model loaded.")

Using device: mps


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded.


## Embed all items

This loops over every matched item image, embeds it the same way as the board images in notebook 02, and collects the results. A quick local benchmark (200 images) measured ~14ms/image on this machine's GPU, so the full ~44k should take roughly 10 minutes.

Two things this loop does differently from notebook 02, both because we're now processing tens of thousands of real downloaded files instead of 10:
- **Progress logging** every 2000 images, with an estimated time remaining, so a 10-minute run doesn't look frozen.
- **Skips unreadable images** instead of crashing the whole run -- datasets this size sometimes contain a handful of corrupt or truncated files, and we'd rather lose a few items than lose 9 minutes of progress to one bad file.

Note we do **not** average these into one vector like we did for the vibe vector -- we keep every item's embedding separately, since Step 6 needs to compare the vibe vector against each one individually.

In [8]:
import time
from PIL import Image

all_embeddings = []
valid_ids = []
skipped = []

start = time.time()

for i, (item_id, path) in enumerate(zip(item_ids, item_paths)):
    try:
        image = Image.open(path).convert("RGB")
    except Exception as e:
        skipped.append((item_id, str(e)))
        continue

    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    embedding = outputs.pooler_output[0]

    all_embeddings.append(embedding)
    valid_ids.append(item_id)

    if (i + 1) % 2000 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(item_paths) - (i + 1)) / rate
        print(f"{i + 1}/{len(item_paths)} processed "
              f"({elapsed:.0f}s elapsed, ~{remaining / 60:.1f} min remaining)")

item_embeddings = torch.stack(all_embeddings)
item_ids = valid_ids

print(f"\nDone. Final shape: {item_embeddings.shape}")
if skipped:
    print(f"Skipped {len(skipped)} unreadable images: {skipped[:5]}")

2000/44419 processed (25s elapsed, ~8.7 min remaining)
4000/44419 processed (49s elapsed, ~8.2 min remaining)
6000/44419 processed (73s elapsed, ~7.8 min remaining)
8000/44419 processed (97s elapsed, ~7.4 min remaining)
10000/44419 processed (121s elapsed, ~6.9 min remaining)
12000/44419 processed (146s elapsed, ~6.6 min remaining)
14000/44419 processed (170s elapsed, ~6.2 min remaining)
16000/44419 processed (194s elapsed, ~5.8 min remaining)
18000/44419 processed (219s elapsed, ~5.3 min remaining)
20000/44419 processed (243s elapsed, ~4.9 min remaining)
22000/44419 processed (267s elapsed, ~4.5 min remaining)
24000/44419 processed (291s elapsed, ~4.1 min remaining)
26000/44419 processed (316s elapsed, ~3.7 min remaining)
28000/44419 processed (340s elapsed, ~3.3 min remaining)
30000/44419 processed (364s elapsed, ~2.9 min remaining)
32000/44419 processed (388s elapsed, ~2.5 min remaining)
34000/44419 processed (412s elapsed, ~2.1 min remaining)
36000/44419 processed (437s elapsed, ~1

## Save the item embeddings

We save two aligned NumPy arrays: the embedding matrix, and the item IDs in the same row order. Keeping them as two separate files (rather than one combined structure) mirrors how we'll actually use them later -- load the matrix for fast vectorized similarity math, and use the ID array only when we need to look up which item a given row corresponds to.

In [ ]:
import numpy as np

np.save(Path("../data/embeddings/item_embeddings.npy"), item_embeddings.cpu().numpy())
np.save(Path("../data/embeddings/item_ids.npy"), np.array(item_ids))

print(f"Saved {len(item_ids)} item embeddings to data/embeddings/")

In [6]:
existing_image_ids = {int(p.stem) for p in images_dir.glob("*.jpg")}

df = df[df["id"].isin(existing_image_ids)].reset_index(drop=True)
print("Rows with a matching image file:", df.shape[0])

item_ids = df["id"].tolist()
item_paths = [images_dir / f"{item_id}.jpg" for item_id in item_ids]

Rows with a matching image file: 44419


In [7]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "patrickjohncyh/fashion-clip"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("Model loaded.")

Using device: mps


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded.


In [9]:
import numpy as np

np.save(Path("../data/embeddings/item_embeddings.npy"), item_embeddings.cpu().numpy())
np.save(Path("../data/embeddings/item_ids.npy"), np.array(item_ids))

print(f"Saved {len(item_ids)} item embeddings to data/embeddings/")

Saved 44419 item embeddings to data/embeddings/
